# 序列化 · 10 维（CHA_1.5A 定版）→ seq_dataset

从干净的定版 10 维宽表 `check_up_特征宽表_10维_v7.csv`（三 SOC 共同工况 CHA_1.5A）切成可喂 seq-to-seq 的张量。

**决定（不变）：**
- **全深度**（不卡 74%，到 50.5%）；深段容量含 G2 倍率不一致，属已知 limitation，只影响 SOH 头、不影响 R。
- **递增窗口（train）**：每块电芯从每个前缀预测其余（观测 2→其余、3→其余…），榨多条样本。
- **固定早期窗口（val/test）**：每块电芯一条，观测前 40%（`EVAL_OBS_FRAC`）、预测其余；干净可比、长电芯不被过度加权。
- **缩放 train-only**：只在 train 的原始 check-up 上 fit（R 先 log1p，SOH 不 log）。
- **mask 张量**：presence（特征缺口，本版覆盖率 73.2%）+ padding + 目标缺失。

**维度：** 每步 10 维 = SOH(1) + 3 SOC × {R0,R1,R2}(9)。横轴 = check-up 序号，纵轴 = [SOH/容量 + R]。4 任务头 = SOH + R0 + R1 + R2。

> **诚实标注：** 递增窗口是标准滑窗增广；同一电芯多个前缀是同一轨迹的重叠子序列，训练窗口数达数千，但独立轨迹仍只有 205（A-train）。防泄漏靠电芯级划分。

In [1]:
import pandas as pd, numpy as np, json
from collections import Counter

IN_WIDE  = "check_up_特征宽表_10维_v7.csv"   # 定版 CHA_1.5A 10维
OUT_NPZ  = "seq_dataset_10dim_v7.npz"
OUT_SCALER = "scaler_10dim_v7.json"
OUT_MANIFEST = "seq_manifest_10dim_v7.csv"

SOH_MIN = None          # 全深度
MIN_OBS = 2
EVAL_OBS_FRAC = 0.4
LOG_R = True
SEED = 42

wide = pd.read_csv(IN_WIDE)
FEAT = [c for c in wide.columns if c.startswith('SOC')]   # 9 个 R
FEATURES = ['SOH'] + FEAT                                  # 第0维=SOH(容量)
R_COLS = FEAT; D = len(FEATURES)
print("宽表:", wide.shape, "| 维度 D =", D, "| 特征:", FEATURES)
print("电芯", wide['Battery_ID'].nunique(), "| split:",
      wide.groupby('split')['Battery_ID'].nunique().to_dict())

宽表: (7155, 15) | 维度 D = 10 | 特征: ['SOH', 'SOC10_R0', 'SOC50_R0', 'SOC90_R0', 'SOC10_R1', 'SOC50_R1', 'SOC90_R1', 'SOC10_R2', 'SOC50_R2', 'SOC90_R2']
电芯 302 | split: {'test': 45, 'train': 212, 'val': 45}


## 1 · 按电芯组装有序序列（SOH 降序 = check-up 序号）+ presence mask

In [2]:
if SOH_MIN is not None:
    wide = wide[wide['SOH'] >= SOH_MIN].copy()
wide = wide.sort_values(['Battery_ID','SOH'], ascending=[True,False]).reset_index(drop=True)

seq = {}
for bid, g in wide.groupby('Battery_ID'):
    vals = g[FEATURES].to_numpy(float)      # [L,D]，缺口 NaN
    seq[bid] = dict(vals=vals, pres=~np.isnan(vals), split=g['split'].iloc[0])
Ls = {b: s['vals'].shape[0] for b,s in seq.items()}
print("序列长度 min/median/max:", min(Ls.values()), int(np.median(list(Ls.values()))), max(Ls.values()))
print("可切样本电芯(L>=MIN_OBS+1):", sum(v>=MIN_OBS+1 for v in Ls.values()))

序列长度 min/median/max: 4 26 31
可切样本电芯(L>=MIN_OBS+1): 302


## 2 · 切窗口
- train：递增（obs 从 MIN_OBS 到 L−1，各预测其余）
- val/test：固定早期窗口（观测前 EVAL_OBS_FRAC，一条/电芯）

In [3]:
samples=[]
for bid,s in seq.items():
    L=s['vals'].shape[0]
    if L<MIN_OBS+1: continue
    if s['split']=='train':
        for ol in range(MIN_OBS,L):
            samples.append(dict(bid=bid,split='train',obs=np.arange(ol),pred=np.arange(ol,L)))
    else:
        ol=min(max(MIN_OBS,int(round(EVAL_OBS_FRAC*L))),L-1)
        samples.append(dict(bid=bid,split=s['split'],obs=np.arange(ol),pred=np.arange(ol,L)))
print("样本数 by split:", dict(Counter(x['split'] for x in samples)))
print("独立电芯 by split:",
      {sp:len({x['bid'] for x in samples if x['split']==sp}) for sp in ['train','val','test']})

样本数 by split: {'train': 4581, 'val': 45, 'test': 45}
独立电芯 by split: {'train': 212, 'val': 45, 'test': 45}


## 3 · train-only 缩放（R 先 log1p），在原始 check-up 上 fit

In [4]:
tr =np.vstack([seq[b]['vals'] for b,s in seq.items() if s['split']=='train'])
trp=np.vstack([seq[b]['pres'] for b,s in seq.items() if s['split']=='train'])
def tf(m):
    o=m.copy()
    if LOG_R:
        for j,f in enumerate(FEATURES):
            if f in R_COLS: o[:,j]=np.log1p(o[:,j])
    return o
tt=tf(tr); mean=np.full(D,np.nan); std=np.full(D,np.nan)
for j in range(D):
    col=tt[:,j][trp[:,j]]; mean[j]=np.nanmean(col); std[j]=np.nanstd(col)+1e-8
json.dump(dict(features=FEATURES,r_cols=R_COLS,log_r=LOG_R,mean=mean.tolist(),std=std.tolist()),
          open(OUT_SCALER,'w'),ensure_ascii=False,indent=2)
scale=lambda m:(tf(m)-mean)/std
print("scaler 已存。前4维 mean/std:")
for f,m,sd in list(zip(FEATURES,mean,std))[:4]: print(f"  {f:10s} mean={m:.3f} std={sd:.3f}")

scaler 已存。前4维 mean/std:
  SOH        mean=78.514 std=10.498
  SOC10_R0   mean=3.284 std=0.219
  SOC50_R0   mean=3.211 std=0.234
  SOC90_R0   mean=3.412 std=0.259


## 4 · 组装 padded 张量 + mask，并导出

In [5]:
N=len(samples); mo=max(len(x['obs'])for x in samples); mp=max(len(x['pred'])for x in samples)
X=np.zeros((N,mo,D),np.float32); Xf=np.zeros((N,mo,D),bool); Xt=np.zeros((N,mo),bool)
Y=np.zeros((N,mp,D),np.float32); Ym=np.zeros((N,mp,D),bool)
for i,x in enumerate(samples):
    s=seq[x['bid']]
    vo,po=s['vals'][x['obs']],s['vals'][x['pred']]; pvo,ppo=s['pres'][x['obs']],s['pres'][x['pred']]
    so,sp_=scale(vo),scale(po); so[~pvo]=0; sp_[~ppo]=0
    lo,lp=len(x['obs']),len(x['pred'])
    X[i,:lo],Xf[i,:lo],Xt[i,:lo]=so,pvo,True; Y[i,:lp],Ym[i,:lp]=sp_,ppo

spa=np.array([x['split']for x in samples]); bida=np.array([x['bid']for x in samples])
packs={sp:{'X':X[spa==sp],'X_fmask':Xf[spa==sp],'X_tmask':Xt[spa==sp],
           'Y':Y[spa==sp],'Y_mask':Ym[spa==sp],'bid':bida[spa==sp]} for sp in['train','val','test']}
np.savez_compressed(OUT_NPZ,features=np.array(FEATURES),
    **{f"{sp}_{k}":v for sp,d in packs.items() for k,v in d.items()})
pd.DataFrame([dict(sample_id=i,Battery_ID=x['bid'],split=x['split'],
                   obs_len=len(x['obs']),pred_len=len(x['pred'])) for i,x in enumerate(samples)]
             ).to_csv(OUT_MANIFEST,index=False,encoding='utf-8-sig')

print("已导出:", OUT_NPZ, "|", OUT_SCALER, "|", OUT_MANIFEST)
print("张量: X",X.shape,"Y",Y.shape)
print("目标有效率(非padding非缺口): %.1f%%" % (100*Ym.sum()/(Ym.shape[0]*Ym.shape[1]*D)))
print("\n=== 小结 ===")
for sp in ['train','val','test']:
    print(f"  {sp:5s}: {(spa==sp).sum():5d} 样本 / {len(set(bida[spa==sp]))} 独立电芯")
print("  每步 10 维 (SOH + 9 R, CHA_1.5A) | 全深度 | train递增 / val,test固定40%窗口")

已导出: seq_dataset_10dim_v7.npz | scaler_10dim_v7.json | seq_manifest_10dim_v7.csv
张量: X (4671, 30, 10) Y (4671, 29, 10)
目标有效率(非padding非缺口): 39.0%

=== 小结 ===
  train:  4581 样本 / 212 独立电芯
  val  :    45 样本 / 45 独立电芯
  test :    45 样本 / 45 独立电芯
  每步 10 维 (SOH + 9 R, CHA_1.5A) | 全深度 | train递增 / val,test固定40%窗口


---
### 产物
`seq_dataset_10dim_v7.npz`（模型直接输入）、`scaler_10dim_v7.json`、`seq_manifest_10dim_v7.csv`。

### 下一步：单任务基线
先分别预测 SOH / R0 / R1 / R2（单头小模型），在 val 上看基础误差，回答"ML 能不能预测"，为 MTL 提供对照。
维度→任务头映射（features 数组给顺序）：第0维=SOH；R0=1–3、R1=4–6、R2=7–9（SOC10/50/90）。